# Imports

In [6]:
from src.get_gender_data import get_founder_gender
from src.scrape_wiki import scrape_wiki_table
from src.get_gender_data import get_founder_gender, _parse_names
from src.filter_df import filter_df

import pandas as pd
import os

# Functions

In [7]:
def rescue_unknowns(df, founder_col, gender_col):
    """Re-process only rows where gender is unknown or contains unknown."""
    mask = df[gender_col].apply(lambda x: 'unknown' in str(x).lower() if x is not None else True)
    unknown_count = mask.sum()

    print(f"Attempting to rescue {unknown_count} unknowns in {gender_col}...")

    # Targeted update
    df.loc[mask, gender_col] = df.loc[mask, founder_col].apply(lambda x: get_founder_gender(x, use_web_search=True))

    new_unknown_count = df[gender_col].apply(lambda x: 'unknown' in str(x).lower()).sum()
    print(f"Resolution complete. Unknowns remaining: {new_unknown_count} (Rescued {unknown_count - new_unknown_count})")
    return df

In [8]:


# Ensure output directory exists
os.makedirs("processed_data", exist_ok=True)

def all_unknown_genders(founder_string):
    """Skip gender prediction entirely and return 'unknown' for every founder."""
    if founder_string is None or (isinstance(founder_string, float) and pd.isna(founder_string)):
        return []
    return ['unknown'] * len(_parse_names(str(founder_string).strip()))

In [9]:
url = "https://en.wikipedia.org/wiki/List_of_unicorn_startup_companies"
idx = 2  # Changed to target the main list of unicorn companies

currentUnicorns_df = scrape_wiki_table(url, idx, "current_unicorns.csv")
currentUnicorns_df.to_csv("current_unicorns.csv")

pastUnicorns_df = scrape_wiki_table(url, idx+1, "past_unicorns.csv")
pastUnicorns_df.to_csv("past_unicorns.csv")

Found 4 wikitable(s)
Saved 619 rows to current_unicorns.csv
Found 4 wikitable(s)
Saved 207 rows to past_unicorns.csv


In [10]:
currentUnicorns_df = pd.read_csv("current_unicorns.csv", index_col=0)
pastUnicorns_df = pd.read_csv("past_unicorns.csv", index_col=0)



In [11]:
print("Predicting genders for current unicorns...")

currentUnicorns_df['Founder_Genders'] = filter_df(currentUnicorns_df, "Founder(s)", all_unknown_genders)

# Save intermediate result
currentUnicorns_df.to_csv("processed_data/current_unicorns_with_gender.csv", index=False)

df_companies_founders_gender = currentUnicorns_df[['Company', 'Founder_Genders', 'Industry']].copy()
# Note: get_founder_gender now returns normalized 'male', 'female', 'unknown'
print("Done.")

Predicting genders for current unicorns...
Done.


In [12]:
display(currentUnicorns_df)
display(pastUnicorns_df)

,Company,Valuation(US$ billions),Valuation date,Industry,Country/countries,Founder(s),Founder_Genders
0,Anthropic,965,May 2026(2026-05)[20],Artificial Intelligence,United States,"Dario Amodei,Daniela Amodei,Jared Kaplan, Jack...","[unknown, unknown, unknown, unknown, unknown, ..."
1,OpenAI,852,March 2026(2026-03)[21],Artificial intelligence,United States,"Sam Altman,Elon Musk,Greg Brockman,Ilya Sutskever","[unknown, unknown, unknown, unknown]"
2,ByteDance,600,April 2026[22],Internet,China,"Zhang Yiming, Liang Rubo","[unknown, unknown]"
3,Stripe,159,February 2026(2026-02)[23],Financial services,United States and Ireland,PatrickandJohn Collison,"[unknown, unknown]"
4,Databricks,134,December 2025(2025-12)[24],Software,United States,"Ali Ghodsi,Andy Konwinski,Ion Stoica,Reynold X...","[unknown, unknown, unknown, unknown, unknown]"
...,...,...,...,...,...,...,...
614,HMD Global,1+,August 2020[570],Mobile Devices,Finland,Jean-Francois Baril,[unknown]
615,IQM,1,July 2022[571],Quantum Computing,Finland,"Jan Goetz, Mikko Möttönen, Kuan Yen Tan, Juha ...","[unknown, unknown, unknown, unknown]"
616,Hostaway,1,December 2024[572],Vacation rental software platform,Finland,"Marcus Räder, Saber Kordestanchi, Mikko Nurminen","[unknown, unknown, unknown]"
617,Papara,1,July 2023[573],Fintech,Turkey,Ahmed Faruk Karslı,[unknown]


,Company,Last valuation(US$billions),Valuation date,Exit date,Exit reason,Exit valuation(US$billions),Country,Founders,col_8
0,SpaceX,1250,February 2026(2026-02)[576],June 2026(2026-06)[577],IPO,1770,United States,Elon Musk,NaN
1,Uber,72,August 2018[578],May 2019[579],IPO,82.4,United States,"Travis Kalanick,Garett Camp",NaN
2,DiDi,62,July 2019[580],June 2021[581],IPO,73,China,Cheng Wei,NaN
3,Facebook,50,January 2011,May 2012[582],IPO,104,United States,"Mark Zuckerberg,Eduardo Saverin, Andrew McColl...",NaN
4,Xiaomi,45,April 2015,July 2018[583],IPO,70,China,Lei Jun,NaN
...,...,...,...,...,...,...,...,...,...
202,Zimi,1+,February 2015[184],March 2021[837],Acquired,0.4,China,NaN,NaN
203,QingCloud,1+,June 2017[184],March 2021[838],IPO,0.46,China,NaN,NaN
204,Novogene,1+,November 2016[184],April 2021,IPO,NaN,China,NaN,NaN
205,MissFresh,1+,December 2017[184],June 2021[839],IPO,2.5,China,NaN,NaN


will run for several minutes:

In [13]:
currentUnicorns_df['Founder_Genders'] = filter_df(currentUnicorns_df, "Founder(s)", all_unknown_genders)


In [14]:
print("Processing Current Unicorns...")
currentUnicorns_df['Founder_Genders'] = filter_df(currentUnicorns_df, "Founder(s)", all_unknown_genders)
currentUnicorns_df = rescue_unknowns(currentUnicorns_df, 'Founder(s)', 'Founder_Genders')

print("\nProcessing Past Unicorns...")
pastUnicorns_df['Founder_Genders'] = filter_df(pastUnicorns_df, "Founders", all_unknown_genders)
pastUnicorns_df = rescue_unknowns(pastUnicorns_df, 'Founders', 'Founder_Genders')

# Save the improved data
currentUnicorns_df.to_csv('processed_data/current_unicorns.csv', index=False)
pastUnicorns_df.to_csv('processed_data/past_unicorns.csv', index=False)

Processing Current Unicorns...
Attempting to rescue 208 unknowns in Founder_Genders...
Resolution complete. Unknowns remaining: 0 (Rescued 208)

Processing Past Unicorns...
Attempting to rescue 76 unknowns in Founder_Genders...
Resolution complete. Unknowns remaining: 0 (Rescued 76)


# Diagramme

In [38]:
mask = (currentUnicorns_df['Founder_Genders'].apply(lambda x: len(x) > 0))
currentUnicorns_with_gender_df = currentUnicorns_df.loc[mask]

mask = (pastUnicorns_df['Founder_Genders'].apply(lambda x: len(x) > 0))
pastUnicorns_with_gender_df = pastUnicorns_df.loc[mask]

mask = (currentUnicorns_df['Founder_Genders'].apply(lambda x: len(x) == 0))
currentUnicorns_without_gender_df = currentUnicorns_df.loc[mask]

mask = (pastUnicorns_df['Founder_Genders'].apply(lambda x: len(x) == 0))
pastUnicorns_without_gender_df = pastUnicorns_df.loc[mask]


In [39]:
display(currentUnicorns_with_gender_df)
display(currentUnicorns_without_gender_df)

display(pastUnicorns_with_gender_df)
display(pastUnicorns_without_gender_df)


,Company,Valuation(US$ billions),Valuation date,Industry,Country/countries,Founder(s),Founder_Genders
0,Anthropic,965,May 2026(2026-05)[20],Artificial Intelligence,United States,"Dario Amodei,Daniela Amodei,Jared Kaplan, Jack...","[male, female, male, male, male, male]"
1,OpenAI,852,March 2026(2026-03)[21],Artificial intelligence,United States,"Sam Altman,Elon Musk,Greg Brockman,Ilya Sutskever","[male, male, male, male]"
2,ByteDance,600,April 2026[22],Internet,China,"Zhang Yiming, Liang Rubo","[male, male]"
3,Stripe,159,February 2026(2026-02)[23],Financial services,United States and Ireland,PatrickandJohn Collison,"[male, male]"
4,Databricks,134,December 2025(2025-12)[24],Software,United States,"Ali Ghodsi,Andy Konwinski,Ion Stoica,Reynold X...","[male, male, male, male, male]"
...,...,...,...,...,...,...,...
614,HMD Global,1+,August 2020[570],Mobile Devices,Finland,Jean-Francois Baril,[male]
615,IQM,1,July 2022[571],Quantum Computing,Finland,"Jan Goetz, Mikko Möttönen, Kuan Yen Tan, Juha ...","[male, male, male, male]"
616,Hostaway,1,December 2024[572],Vacation rental software platform,Finland,"Marcus Räder, Saber Kordestanchi, Mikko Nurminen","[male, male, male]"
617,Papara,1,July 2023[573],Fintech,Turkey,Ahmed Faruk Karslı,[male]


,Company,Valuation(US$ billions),Valuation date,Industry,Country/countries,Founder(s),Founder_Genders
26,Rippling,16.8,May 2025[42],Workforce management,United States,NaN,[]
61,WHOOP,10.1,March 2026(2026-03)[78],Wearable technology,United States,NaN,[]
62,Chehaoduo,10,July 2021[79],Marketplace,China,NaN,[]
63,Digital Currency Group,10,November 2021[80],Venture capital,United States,NaN,[]
66,KuCoin,10,May 2022[83],Cryptocurrency,Seychelles,NaN,[]
...,...,...,...,...,...,...,...
602,LINE MAN Wongnai,1+,September 2022[184],"E-commerce,Food delivery",Thailand,NaN,[]
604,DANA,1+,September 2022,Financial technology,Indonesia,NaN,[]
606,Ecovadis,1,August 2022[128],"Environmental, social, and corporate governance",France,NaN,[]
609,Payhawk,1+,March 2022[566],Financial technology,Bulgaria,NaN,[]


,Company,Last valuation(US$billions),Valuation date,Exit date,Exit reason,Exit valuation(US$billions),Country,Founders,col_8,Founder_Genders
0,SpaceX,1250,February 2026(2026-02)[576],June 2026(2026-06)[577],IPO,1770,United States,Elon Musk,NaN,[male]
1,Uber,72,August 2018[578],May 2019[579],IPO,82.4,United States,"Travis Kalanick,Garett Camp",NaN,"[male, male]"
2,DiDi,62,July 2019[580],June 2021[581],IPO,73,China,Cheng Wei,NaN,[male]
3,Facebook,50,January 2011,May 2012[582],IPO,104,United States,"Mark Zuckerberg,Eduardo Saverin, Andrew McColl...",NaN,"[male, male, male, male, male]"
4,Xiaomi,45,April 2015,July 2018[583],IPO,70,China,Lei Jun,NaN,[male]
...,...,...,...,...,...,...,...,...,...,...
168,CRF Health,1,July 2018,July 2018,Acquisition[798],1,Finland,"Timo Ahopelto, Jarkko Joki-Tokola, Jaakko Ollila",NaN,"[male, male, male]"
169,MySQL,1,January 2008,January 2008,Acquisition[799],1,Finland,Michael Widenius,NaN,[male]
170,Heptagon,1,February 2018,February 2018,Acquisition[800],1,Finland,Christian Tang-Jespersen,NaN,[male]
172,Shopify,1,December 2013,May 2015,IPO,1.3,Canada,"Tobias Lütke, Scott Lake",NaN,"[male, male]"


,Company,Last valuation(US$billions),Valuation date,Exit date,Exit reason,Exit valuation(US$billions),Country,Founders,col_8,Founder_Genders
6,Lufax,39.4,March 2019,October 2020[585],IPO,33,China,NaN,NaN,[]
10,Meituan-Dianping,30,October 2017[184],September 2018[591],IPO,53,China,NaN,NaN,[]
13,Coreweave,23,March 2025[595],March 2025[595],NaN,23,United States,NaN,NaN,[]
15,CATL,20.0,March 2018[596],June 2018[597],IPO,12.3,China,NaN,NaN,[]
16,Tencent Music,18,January 2018,December 2018[598],IPO,21.3,China,NaN,NaN,[]
...,...,...,...,...,...,...,...,...,...,...
202,Zimi,1+,February 2015[184],March 2021[837],Acquired,0.4,China,NaN,NaN,[]
203,QingCloud,1+,June 2017[184],March 2021[838],IPO,0.46,China,NaN,NaN,[]
204,Novogene,1+,November 2016[184],April 2021,IPO,NaN,China,NaN,NaN,[]
205,MissFresh,1+,December 2017[184],June 2021[839],IPO,2.5,China,NaN,NaN,[]


In [49]:
currentUnicorns_with_gender_df['Valuation(US$ billions)'] = pd.to_numeric(
    currentUnicorns_with_gender_df['Valuation(US$ billions)'],
    errors='coerce'   # wrong values -> NaN
)
current_unicorn_w_gender_valuation = currentUnicorns_with_gender_df['Valuation(US$ billions)'].astype(float).sum()

currentUnicorns_without_gender_df['Valuation(US$ billions)'] = pd.to_numeric(
    currentUnicorns_without_gender_df['Valuation(US$ billions)'],
    errors='coerce'   # wrong values -> NaN
)
current_unicorn_wo_gender_valuation = currentUnicorns_without_gender_df['Valuation(US$ billions)'].astype(float).sum()





pastUnicorns_with_gender_df['Last valuation(US$billions)'] = pd.to_numeric(
    pastUnicorns_with_gender_df['Last valuation(US$billions)'],
    errors='coerce'   # wrong values -> NaN
)
past_unicorn_w_gender_valuation = pastUnicorns_with_gender_df['Last valuation(US$billions)'].astype(float).sum()

pastUnicorns_without_gender_df['Last valuation(US$billions)'] = pd.to_numeric(
    pastUnicorns_without_gender_df['Last valuation(US$billions)'],
    errors='coerce'   # wrong values -> NaN
)
past_unicorn_wo_gender_valuation = pastUnicorns_without_gender_df['Last valuation(US$billions)'].astype(float).sum()

print(current_unicorn_w_gender_valuation, current_unicorn_wo_gender_valuation, past_unicorn_w_gender_valuation, past_unicorn_wo_gender_valuation)

4243.795 849.0200000000001 2067.17 379.78
